**Run `harvest_kw.ipynb` first**, and have an Aurora kW CSV with matching timestamps.


## 3. Compare Harvest kW to Aurora kW

Joins the two 15-minute kW files on meter name and datetime, writes a PDF (one chart per meter), and a score table (ok / zeros / missing, correlation, percent difference).

**Column names:** Harvest uses `mean_kw`; Aurora uses `mean` (or `blue_pillar_kw`, which gets renamed).

**You should get:** a plots PDF under `../data/outputs/plots/` and a `*_comparison_info.csv`. Optionally a merged CSV.


### Enter input

- **`harvest_csv` / `aurora_csv`** — both kW files. Datetimes must already line up (this notebook does not snap times).
- **`create_merged_csv`** — also save the joined table.
- **`corr_threshold` / `pct_threshold`** — how strict “match = yes” is (defaults 0.95 and 10%).
- Create `plot_dir` on disk if it does not exist, or the PDF save will fail.


In [ ]:
# directories
output_dir = '../data/outputs/'
plot_dir = '../data/outputs/plots/'

# harvest input csv
harvest_csv = output_dir + 'harvest_kw_250907-250909.csv' #'harvest_kw_######-######.csv'

# aurora input csv
aurora_csv = output_dir +'aurora_kw_250907-250909.csv'

##########################################################################

# True if create csv of merged data
create_merged_csv = False

# correlation threshold for comparison info 
corr_threshold = 0.95

# percentage difference threshold for comparison info
pct_threshold = 10.0

##########################################################################

# data comparison name
name = 'harvest_aurora' # for file naming

# merged csv of both datasets

# pdf of comparison plots

# comparison info csv



### Imports


In [2]:
%load_ext autoreload
%autoreload 2

import os, sys
sys.path.append(os.path.abspath('..'))
import modules.harvest_kw_comp as hv_kw # import self defined module
import modules.file_naming as fn # import self defined module

### Merge, plot, and score

`load_data_for_comparison` → optional merged CSV → `create_plots_pdf` → `get_comparison_info`.

In [ ]:
# create merged dataframe of the input csvs and get the list of meters
merged_df, meters = hv_kw.load_data_for_comparison(harvest_csv, aurora_csv)

# var names for file naming
var1 = 'plots'
var2 = 'comparison_info'
var3 = 'merged'

# optional create csv of merged dataframe
if create_merged_csv:
    # create merged filename
    merged_filename = output_dir + fn.make_filename(merged_df, name, var3, 'csv')

    merged_df.to_csv(merged_filename, index=False)

# create plots filename
plots_filename = plot_dir + fn.make_filename(merged_df, name, var1, 'pdf')

# create pdf of plots comparing harvest vs aurora kw data per meter
hv_kw.create_plots_pdf(merged_df, meters, plots_filename)

# create dataframe of info summarizing comparison harvest vs aurora kw data per meter
info_df = hv_kw.get_comparison_info(merged_df, meters, corr_threshold, pct_threshold)

# create comparison_info filename
comparison_info_filename = output_dir + fn.make_filename(merged_df, name, var2, 'csv')

# create csv of comparison info dataframe
info_df.to_csv(comparison_info_filename, index=True)